In [ ]:
# Notebook-baked configuration (substituted at notebook creation time)
workspace_id = "__SPINDLE_WORKSPACE_ID__"
lakehouse_id = "__SPINDLE_LAKEHOUSE_ID__"
schema_path = "spindle_temp/current_run.json"


In [ ]:
# Install sqllocks-spindle from PyPI (Phase 1 + Phase 2 features)
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
     "sqllocks-spindle[fabric-files]==2.7.3"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed: {r.stderr}")
print("sqllocks-spindle 2.7.1 installed from PyPI")


In [ ]:
# Read schema JSON via Spark (cluster already has authenticated abfss access)
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
abfss_root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
schema_uri = f"{abfss_root}/Files/{schema_path}"

# wholeTextFiles returns RDD of (path, content) — reads full file as one string
schema_raw = spark.sparkContext.wholeTextFiles(schema_uri).collect()[0][1]
schema_dict = json.loads(schema_raw)

_rt = schema_dict.get("_runtime", {})
chunk_size = int(_rt.get("chunk_size", 500_000))
seed = int(_rt.get("seed", 42))
total_rows = int(_rt.get("total_rows", 1000))
sink_names = _rt.get("sinks", ["lakehouse"])
sink_cfg = _rt.get("sink_config", {})
print(f"Loaded {len(schema_raw):,} bytes; total_rows={total_rows:,}, sinks={sink_names}")
print(f"Static tables: {schema_dict.get('_static_tables', [])}")
print(f"Dynamic tables: {schema_dict.get('_dynamic_tables', [])}")


In [ ]:
# Write tables as Delta to /Tables/ via Spark (cluster has authenticated abfss access)
import math
import json
import os
import tempfile
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
tables_root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

def write_delta(pandas_df, table_name):
    if pandas_df.empty:
        print(f"  skip {table_name}: empty")
        return
    sdf = spark.createDataFrame(pandas_df)
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{tables_root}/spindle_{table_name}")
    print(f"  wrote {table_name}: {len(pandas_df):,} rows")

# --- Static tables: already generated, in schema_dict["_static_pk_data"] ---
_static_pk_data = schema_dict.get("_static_pk_data", {})
print(f"Writing {len(_static_pk_data)} static tables as Delta to /Tables/...")
for tname, col_lists in _static_pk_data.items():
    pdf = pd.DataFrame(col_lists)
    write_delta(pdf, tname)

# --- Dynamic tables: generate on driver (foreachPartition disabled for now) ---
dynamic_tables = schema_dict.get("_dynamic_tables", [])
if dynamic_tables:
    from sqllocks_spindle.engine.chunk_worker import generate_chunk
    n_chunks = max(1, math.ceil(total_rows / chunk_size))
    print(f"Generating {n_chunks} chunk(s) for dynamic tables: {dynamic_tables}")

    with tempfile.NamedTemporaryFile(suffix=".json", mode="w", delete=False) as f:
        json.dump(schema_dict, f)
        tmp_path = f.name

    try:
        accumulated = {t: {} for t in dynamic_tables}
        for i in range(n_chunks):
            chunk_offset = i * chunk_size
            chunk_count = min(chunk_size, total_rows - chunk_offset)
            if chunk_count <= 0:
                continue
            chunk_seed = seed ^ i
            chunk_data = generate_chunk(
                schema_path=tmp_path, seed=chunk_seed,
                offset=chunk_offset, count=chunk_count,
            )
            print(f"  chunk {i+1}/{n_chunks}: {chunk_count:,} rows generated")
            for t in dynamic_tables:
                if t not in chunk_data:
                    continue
                for col, vals in chunk_data[t].items():
                    accumulated[t].setdefault(col, []).extend(vals)

        for t in dynamic_tables:
            if accumulated[t]:
                pdf = pd.DataFrame(accumulated[t])
                write_delta(pdf, t)
    finally:
        os.unlink(tmp_path)
else:
    print("No dynamic tables.")

print("All tables written.")


In [ ]:
# Write result stats to OneLake and clean up the temp schema file
import json
from notebookutils import mssparkutils

result = {
    "rows_generated": total_rows,
    "status": "succeeded",
}

abfss_root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
result_uri = f"{abfss_root}/Files/spindle_temp/current_run_result.json"
mssparkutils.fs.put(result_uri, json.dumps(result), overwrite=True)
print(f"Result written: {result_uri}")

try:
    schema_uri = f"{abfss_root}/Files/{schema_path}"
    mssparkutils.fs.rm(schema_uri)
    print(f"Cleaned up schema file: {schema_uri}")
except Exception as e:
    print(f"Warning: could not delete schema file: {e}")
